### Data implementation

In [1]:
import torch 
import os
from torch.utils.data import dataloader,Dataset
from torchvision import transforms 
from PIL import Image 

In [2]:
class Imageprocessor:
    def __init__(self,root_dir_path,transformations=None):
        self.root_dir_path=root_dir_path
        self.transformations=transformations
        self.all_image_path=[os.path.join(root_dir_path,img) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_image_path)

    def __getitem__(self,idx):
        img_path=self.self.all_image_path[idx]
        img=Image.open(img_path).convert(RGB)
        if transformations:
            img=self.transformations(img)
        return img 

In [3]:
root_dir_path='./img_align_celeba'
transformations=transforms.Compose([
    transforms.CenterCrop(178),
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
])

### generation network

In [4]:
import torch.nn as nn 
import torch.optim as optim
import numpy as np

In [8]:
class Generator(nn.Module):
    def __init__(self,z_dim=100,img_channels=3):
        super(Generator,self).__init__()
        self.model=nn.Sequential(
            nn.Linear(z_dim,256),
            nn.ReLU(),

            nn.Linear(256,512),
            nn.ReLU(),

            nn.Linear(512,1024),
            nn.ReLU(),

            nn.Linear(1024,64*64*img_channels),
            nn.Tanh(),
            
        )
    def forward(self,z):
        img=self.model(z)
        img=img.view(img.size(0),3,64,64)
        return img

### Discriminator Network

In [9]:
class Discriminator(nn.Module):
    def __init__(self,img_channels=3):
        super(Discriminator,self).__init__()
        self.model=nn.Sequential(
            nn.Flatten(),
            
            nn.Linear(64*64*img_channels,1024),
            nn.LeakyReLU(0.2,inplace=True),

            nn.Linear(1024,512),
            nn.LeakyReLU(0.2,inplace=True),

            nn.Linear(1024,256),
            nn.LeakyReLU(0.2,inplace=True),

            nn.Linear(256,1),
            nn.Sigmoid()
            
        )
    def forward(self,img):
        return self.model(img)


In [10]:
GAN_Loss=nn.BCELoss()
generator=Generator()
g_optimiser=optim.Adam(generator.parameters(),lr=0.0002,betas=(0.5,0.999))
discriminator=Discriminator()
g_optimiser=optim.Adam(discriminator.parameters(),lr=0.0002,betas=(0.5,0.999))

In [11]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


In [12]:
generator=generator.to(device)
discriminator=discriminator.to(device)

### Training GAN